# Track B — 등급표 재계산 + H5 재확인 (track_b_10)

담당: 김태헌 | 선행 노트북: track_b_07(모델 튜닝·선정·씬파일러 스코어링)

## 이 노트북의 목적

track_b_06의 컷오프별 정밀도·재현율 표(1등급/5등급/9등급)와, track_b_05의 H5(씬파일러 등급
분산도, "중간등급 76% 쏠림·균등도 0.90")는 둘 다 **튜닝 전 고정 파라미터 모델** 기준이었다.
이 노트북은 track_b_07에서 선정된 최종 모델(`selected_model`, `track_b_best_params_final.json`)
기준으로 두 가지를 재계산한다.

1. **등급표 재계산**: 9등급 컷오프별 정밀도·재현율 표
2. **H5 재확인**: 씬파일러가 정말 중간등급에 몰리는지, 균등도는 어떻게 변하는지

## 등급 산정 방식

9등급으로 나누되, 기본값은 **동일 인구 비율(각 등급 ≈11.1%)** 분위수 기준이다. 팀이 이전에
KCB식 비균등 등급 구간(예: 1등급 100%, 5등급 40%, 9등급 10%처럼 등급별 인구비중이 다른 방식)을
썼다면, `GRADE_BOUNDARIES`를 직접 지정해서 그 기준에 맞출 수 있다(3-1절 참고).


## 0. 환경 설정

In [11]:
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42


## 1. 데이터 로드 및 최종 모델 재구성 (track_b_07과 동일한 구성)

track_b_07과 완전히 동일한 방식으로 데이터를 불러오고, `track_b_best_params_final.json`에
저장된 최적 파라미터 + 선정된 모델 종류로 다시 학습한다.

In [12]:
handoff_path = r'C:\Users\tehun\Desktop\multicamp\project\creditscore\cardCB'  # 필요시 수정

df_train = pd.read_csv(f'{handoff_path}/track_b_features_train_v2.csv')
df_apply = pd.read_csv(f'{handoff_path}/track_b_features_apply_v2.csv')

categorical_cols = ['JB_TP', 'HOME_ADM']
feature_cols = [c for c in df_train.columns if c not in ['CUST_ID', 'TARGET']]

X_full = df_train[feature_cols]
y_full = df_train['TARGET']

X_full_enc = pd.get_dummies(X_full, columns=categorical_cols, prefix=categorical_cols)
final_feature_cols = X_full_enc.columns.tolist()

X_apply_enc = pd.get_dummies(df_apply[feature_cols], columns=categorical_cols, prefix=categorical_cols)
X_apply_enc = X_apply_enc.reindex(columns=final_feature_cols, fill_value=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_full_enc, y_full, test_size=0.2, stratify=y_full, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


Train: (228712, 90), Test: (57178, 90)


In [13]:
with open(f'{handoff_path}/track_b_best_params_final.json', encoding='utf-8') as f:
    best_record = json.load(f)

selected_model_name = best_record['selected_model']
print(f"track_b_07에서 선정된 최종 모델: {selected_model_name}")

if selected_model_name == 'XGBoost':
    params = best_record['XGBoost']
    best_model = xgb.XGBClassifier(
        **params,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        eval_metric='aucpr', random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist',
    )
    best_model.fit(X_train, y_train)

elif selected_model_name == 'RandomForest':
    params = best_record['RandomForest']
    best_model = RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
    best_model.fit(X_train, y_train)

elif selected_model_name == 'LogisticRegression':
    params = best_record['LogisticRegression']
    # RandomizedSearchCV 파라미터 키가 'clf__C' 형태이므로 파이프라인으로 재구성
    clf_params = {k.replace('clf__', ''): v for k, v in params.items()}
    best_model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=3000, class_weight='balanced',
                                    random_state=RANDOM_STATE, **clf_params))
    ])
    best_model.fit(X_train, y_train)

else:
    raise ValueError(f"알 수 없는 모델명: {selected_model_name}")

print("최종 모델 재학습 완료")


track_b_07에서 선정된 최종 모델: XGBoost
최종 모델 재학습 완료


## 2. 예측 점수 산출 (test + 씬파일러)

In [14]:
proba_test = best_model.predict_proba(X_test)[:, 1]
thin_proba = best_model.predict_proba(X_apply_enc)[:, 1]

print(f"Test 예측 완료: {len(proba_test)}건")
print(f"씬파일러 예측 완료: {len(thin_proba)}건")


Test 예측 완료: 57178건
씬파일러 예측 완료: 44110건


## 3-1. 등급 경계 정의 — ChiMerge(카이제곱 병합)

원 문서(H5) 기준: **ChiMerge**로 만든 9등급. 균등분위와 달리 인구비율을 강제로 맞추지 않고,
인접 구간의 TARGET(연체) 분포가 통계적으로 가장 비슷한 쌍부터 순서대로 병합해서 9개 구간으로
줄인다. 등급 간 경계가 "위험도 차이"를 기준으로 정해지므로, 원 문서의 비균등 등급 체계
(1등급 100% · 5등급 40% · 9등급 10%)를 재현한다. 또한 신용이력보유군의 등급 분포가 더 이상
정의상 균등해지지 않으므로(이전 버전의 순환논리 해소), 균등도 비교가 의미를 갖는다.

`신용이력보유군(test set, proba_test + y_test)` 기준으로만 경계를 학습하고, 씬파일러에는
그 경계를 그대로 적용한다 — 씬파일러는 TARGET이 없으므로 경계 학습에 관여하지 않는다.

In [15]:
def chimerge_fit(score, target, max_bins=9, initial_bins=100):
    """
    ChiMerge(카이제곱 병합) 지도학습 기반 구간화.
    1) score를 initial_bins개의 세밀한 구간으로 우선 쪼갠다(분위수 기준).
    2) 인접한 두 구간의 (양성/음성) 카운트로 2x2 분할표 카이제곱 통계량을 계산.
    3) 카이제곱 값이 가장 작은(=target 분포 차이가 가장 안 유의미한) 인접쌍을 병합.
    4) 구간 수가 max_bins가 될 때까지 3)을 반복.

    Returns: 정렬된 bin edges (길이 max_bins+1, 양끝은 -inf/inf)
    """
    score = np.asarray(score)
    target = np.asarray(target)

    init_edges = np.unique(np.quantile(score, np.linspace(0, 1, initial_bins + 1)))
    init_edges[0], init_edges[-1] = -np.inf, np.inf

    bin_id = np.digitize(score, init_edges[1:-1])
    n_bins = len(init_edges) - 1

    counts = np.zeros((n_bins, 2))
    for b in range(n_bins):
        mask = bin_id == b
        counts[b, 0] = (target[mask] == 0).sum()
        counts[b, 1] = (target[mask] == 1).sum()

    edges = list(init_edges)

    def chi2_adjacent(c1, c2):
        table = np.array([c1, c2])
        row_sums, col_sums = table.sum(axis=1), table.sum(axis=0)
        total = table.sum()
        if total == 0:
            return 0.0
        expected = np.outer(row_sums, col_sums) / total
        with np.errstate(divide='ignore', invalid='ignore'):
            terms = np.where(expected > 0, (table - expected) ** 2 / expected, 0)
        return terms.sum()

    while len(counts) > max_bins:
        chi2_values = [chi2_adjacent(counts[i], counts[i + 1]) for i in range(len(counts) - 1)]
        merge_idx = int(np.argmin(chi2_values))
        counts[merge_idx] = counts[merge_idx] + counts[merge_idx + 1]
        counts = np.delete(counts, merge_idx + 1, axis=0)
        del edges[merge_idx + 1]

    return np.array(edges)


N_GRADES = 9
grade_edges = chimerge_fit(proba_test, y_test.values, max_bins=N_GRADES, initial_bins=100)
print(f"ChiMerge 등급 경계 ({len(grade_edges)-1}개 구간):")
print(grade_edges)

def assign_grade(scores, edges):
    return np.digitize(scores, edges[1:-1]) + 1

grade_test = assign_grade(proba_test, grade_edges)
grade_thin = assign_grade(thin_proba, grade_edges)

print("\n신용이력보유군(test) 등급별 인구비율(%):")
print((pd.Series(grade_test).value_counts(normalize=True).sort_index() * 100).round(1))
print("\n등급 이상 누적 인구비율(%) — 원 문서 형식과 비교:")
for g in range(1, N_GRADES + 1):
    cum_pct = (grade_test >= g).mean() * 100
    print(f"  {g}등급 이상: {cum_pct:.1f}%")


ChiMerge 등급 경계 (9개 구간):
[      -inf 0.03168797 0.11421199 0.23317128 0.3679401  0.49900727
 0.59234013 0.74434165 0.8513383         inf]

신용이력보유군(test) 등급별 인구비율(%):
1    19.0
2    22.0
3    20.0
4    16.0
5    10.0
6     5.0
7     5.0
8     2.0
9     1.0
Name: proportion, dtype: float64

등급 이상 누적 인구비율(%) — 원 문서 형식과 비교:
  1등급 이상: 100.0%
  2등급 이상: 81.0%
  3등급 이상: 59.0%
  4등급 이상: 39.0%
  5등급 이상: 23.0%
  6등급 이상: 13.0%
  7등급 이상: 8.0%
  8등급 이상: 3.0%
  9등급 이상: 1.0%


### 3-2. 등급별 정밀도·재현율 표 (컷오프: N등급 이상 = 위험군)

In [16]:
rows = []
y_test_arr = y_test.values
for cutoff in range(1, N_GRADES + 1):
    pred_risk = (grade_test >= cutoff).astype(int)
    risk_pop_pct = pred_risk.mean() * 100
    if pred_risk.sum() == 0:
        precision, recall = np.nan, np.nan
    else:
        precision = precision_score(y_test_arr, pred_risk, zero_division=0)
        recall = recall_score(y_test_arr, pred_risk, zero_division=0)
    rows.append({
        '컷오프': f'{cutoff}등급 이상',
        '위험군_비율(%)': round(risk_pop_pct, 1),
        '정밀도': round(precision, 4) if precision == precision else None,
        '재현율': round(recall, 4) if recall == recall else None,
    })

grade_table = pd.DataFrame(rows)
grade_table


,컷오프,위험군_비율(%),정밀도,재현율
0,1등급 이상,100.0,0.0423,1.0000
1,2등급 이상,81.0,0.0514,0.9851
2,3등급 이상,59.0,0.0680,0.9487
3,4등급 이상,39.0,0.0938,0.8652
4,5등급 이상,23.0,0.1338,0.7275
5,6등급 이상,13.0,0.1866,0.5736
6,7등급 이상,8.0,0.2411,0.4562
7,8등급 이상,3.0,0.3514,0.2494
8,9등급 이상,1.0,0.5017,0.1187


## 4. H5 재확인: 씬파일러 등급 분산도

track_b_05 원 결과: 씬파일러는 중간등급(2~5등급)에 76% 몰림(신용이력보유군은 60%),
균등도(정규화 엔트로피) 씬파일러 0.90 vs 신용이력보유군 0.95.

In [17]:
def normalized_entropy(grade_array, n_grades):
    counts = pd.Series(grade_array).value_counts().reindex(range(1, n_grades + 1), fill_value=0)
    p = counts / counts.sum()
    p_nonzero = p[p > 0]
    entropy = -(p_nonzero * np.log(p_nonzero)).sum()
    return entropy / np.log(n_grades)  # 0~1로 정규화, 1=완전 균등

MID_GRADES = range(2, 6)  # 2~5등급 정의 (원 문서 기준)

def mid_grade_pct(grade_array):
    return np.isin(grade_array, list(MID_GRADES)).mean() * 100

h5_result = pd.DataFrame([
    {
        'population': '신용이력보유군(test)',
        '중간등급(2~5)_비율(%)': round(mid_grade_pct(grade_test), 1),
        '균등도(정규화엔트로피)': round(normalized_entropy(grade_test, N_GRADES), 4),
    },
    {
        'population': '씬파일러',
        '중간등급(2~5)_비율(%)': round(mid_grade_pct(grade_thin), 1),
        '균등도(정규화엔트로피)': round(normalized_entropy(grade_thin, N_GRADES), 4),
    },
])

print("track_b_05 원 결과(고정 파라미터 기준): 씬파일러 76% / 0.90, 신용이력보유군 60% / 0.95")
print()
h5_result


track_b_05 원 결과(고정 파라미터 기준): 씬파일러 76% / 0.90, 신용이력보유군 60% / 0.95



,population,중간등급(2~5)_비율(%),균등도(정규화엔트로피)
0,신용이력보유군(test),68.0,0.8729
1,씬파일러,79.8,0.8320


## 5. H5 판정

- **중간등급 쏠림 방향과 균등도 대소관계가 유지되면** → H5 "부분적으로만 지지됨" 결론 재확정
- **방향이 바뀌거나 격차가 크게 벌어지면** → 튜닝된 모델에서 씬파일러 스코어링 특성이
  달라졌다는 뜻이므로, 원인(어떤 변수가 씬파일러에게 유독 극단값을 주는지) SHAP로 확인 필요

## 6. 결과 저장

In [18]:
grade_table.to_csv(f'{handoff_path}/track_b_grade_table_final.csv', index=False, encoding='utf-8-sig')
h5_result.to_csv(f'{handoff_path}/track_b_h5_revalidation_final.csv', index=False, encoding='utf-8-sig')

df_thin_grade = df_apply[['CUST_ID']].copy()
df_thin_grade['score_thin_filers'] = thin_proba
df_thin_grade['grade'] = grade_thin
df_thin_grade.to_csv(f'{handoff_path}/track_b_thin_filer_grades_final.csv', index=False, encoding='utf-8-sig')

print("저장 완료: track_b_grade_table_final.csv, track_b_h5_revalidation_final.csv, track_b_thin_filer_grades_final.csv")
print(grade_table)
print()
print(h5_result)


저장 완료: track_b_grade_table_final.csv, track_b_h5_revalidation_final.csv, track_b_thin_filer_grades_final.csv
      컷오프  위험군_비율(%)     정밀도     재현율
0  1등급 이상      100.0  0.0423  1.0000
1  2등급 이상       81.0  0.0514  0.9851
2  3등급 이상       59.0  0.0680  0.9487
3  4등급 이상       39.0  0.0938  0.8652
4  5등급 이상       23.0  0.1338  0.7275
5  6등급 이상       13.0  0.1866  0.5736
6  7등급 이상        8.0  0.2411  0.4562
7  8등급 이상        3.0  0.3514  0.2494
8  9등급 이상        1.0  0.5017  0.1187

      population  중간등급(2~5)_비율(%)  균등도(정규화엔트로피)
0  신용이력보유군(test)             68.0        0.8729
1           씬파일러             79.8        0.8320


In [19]:
print(f"신용이력보유군 test 평균 예측확률: {proba_test.mean():.4f} (실제 연체율 {y_test.mean():.4%})")

신용이력보유군 test 평균 예측확률: 0.2269 (실제 연체율 4.2289%)
